# ShapeVision CNN

Binary image classification with a residual CNN. Run the cells in order. Generate a small circle-versus-square dataset with `python scripts/generate_demo_data.py`, or provide your own class `0` and `1` image folders. The generated images are for a functional demo, not a real-world accuracy benchmark.

In [ ]:
import os
import time
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

torch.backends.cudnn.benchmark = True

CONFIG

In [ ]:
TRAIN_DIR = "data/train/train"
TEST_DIR = "data/test/test"
CHECKPOINT_PATH = "artifacts/best_model.pt"
SUBMISSION_PATH = "artifacts/submission.csv"
IMAGE_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 20
LR = 3e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

DATASET

In [ ]:
class TrainDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            img = self.transform(img)

        return img, label


class TestDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, os.path.basename(self.image_paths[idx])

AUGMENTATION

In [ ]:
train_tfms = v2.Compose([
    v2.RandomResizedCrop((IMAGE_SIZE, IMAGE_SIZE), scale=(0.7, 1.0)),
    v2.RandomHorizontalFlip(),
    v2.RandomVerticalFlip(),
    v2.RandomRotation(degrees=(-15, 15)),
    v2.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.5] * 3, [0.5] * 3),
])

val_tfms = v2.Compose([
    v2.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.5] * 3, [0.5] * 3),
])

LOAD DATA

In [ ]:
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}
image_paths, labels = [], []

for label in (0, 1):
    folder = os.path.join(TRAIN_DIR, str(label))
    if not os.path.isdir(folder):
        raise FileNotFoundError(
            f"Missing {folder}. Add your images or run: python scripts/generate_demo_data.py"
        )
    for name in sorted(os.listdir(folder)):
        path = os.path.join(folder, name)
        if os.path.isfile(path) and os.path.splitext(name)[1].lower() in IMAGE_EXTENSIONS:
            image_paths.append(path)
            labels.append(label)

if min(labels.count(0), labels.count(1)) < 2:
    raise ValueError("Each training class needs at least two supported image files")

validation_count = max(2, int(np.ceil(len(image_paths) * 0.1)))
train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths, labels, test_size=validation_count, stratify=labels, random_state=42
)

train_ds = TrainDataset(train_paths, train_labels, train_tfms)
val_ds = TrainDataset(val_paths, val_labels, val_tfms)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
    pin_memory=DEVICE.type == "cuda"
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
    pin_memory=DEVICE.type == "cuda"
)

print(f"Training images: {len(train_ds)} | Validation images: {len(val_ds)}")
image, label = train_ds[0]
print(f"Dataset ready: image shape {tuple(image.shape)}, label {label}")

CNN

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.relu = nn.ReLU(inplace=True)
        
        self.conv2 = nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride, bias=False),
                nn.BatchNorm2d(out_c)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return self.relu(out)
    
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, 1, 1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )

        self.layer1 = ResidualBlock(32, 64, stride=2)
        self.layer2 = ResidualBlock(64, 128, stride=2)
        self.layer3 = ResidualBlock(128, 256, stride=2)
        self.layer4 = ResidualBlock(256, 512, stride=2)

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        return self.fc(x)


Model

In [ ]:
model = CNN().to(DEVICE)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

TRAINING

In [ ]:
train_losses = []
val_accuracies = []
val_f1s = []

best_f1 = -1.0
beg = time.perf_counter()
for epoch in range(EPOCHS):
    start = time.perf_counter()
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    model.train()
    train_loss = 0

    for i, (imgs, labels) in enumerate(train_loader):
        imgs = imgs.to(DEVICE)
        labels = labels.float().unsqueeze(1).to(DEVICE)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if i % 100 == 0:
            print(f"Batch {i}/{len(train_loader)} | Images processed: {i * BATCH_SIZE}")

        train_loss += loss.item()

    model.eval()
    val_preds, val_targets = [], []

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            preds = (torch.sigmoid(outputs).cpu().numpy().ravel() > 0.5).astype(int)

            val_preds.extend(preds.tolist())
            val_targets.extend(labels.numpy().tolist())

        acc = accuracy_score(val_targets, val_preds)
    f1 = f1_score(val_targets, val_preds, zero_division=0)

    train_losses.append(train_loss / len(train_loader))
    val_accuracies.append(acc)
    val_f1s.append(f1)

    if f1 > best_f1:
        best_f1 = f1
        os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)
        torch.save({"model_state": model.state_dict(), "epoch": epoch + 1, "val_f1": f1}, CHECKPOINT_PATH)

    current_lr = optimizer.param_groups[0]['lr']

    print(f"\nTime taken for Epoch {epoch+1}/{EPOCHS} = {(time.perf_counter()-start)/60} min")
    print(f"LR: {current_lr:.6f}")
    print(f"Train Loss: {train_loss/len(train_loader):.4f}")
    print(f"Val Accuracy: {acc:.4f}")
    print(f"Val F1 Score: {f1:.4f}")
    print("-"*20)

    scheduler.step()

print("="*20)
print(f"\nTotal time taken = {(time.perf_counter()-beg)/60} min")

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(train_losses, label="Train Loss")
plt.legend()
plt.title("Training Loss")
plt.show()

plt.figure()
plt.plot(val_accuracies, label="Val Accuracy")
plt.plot(val_f1s, label="Val F1")
plt.legend()
plt.title("Validation Metrics")

plt.show()

Test

In [ ]:
if not os.path.isdir(TEST_DIR):
    raise FileNotFoundError(f"Missing test folder: {TEST_DIR}")
test_paths = [
    os.path.join(TEST_DIR, name) for name in sorted(os.listdir(TEST_DIR))
    if os.path.isfile(os.path.join(TEST_DIR, name))
    and os.path.splitext(name)[1].lower() in IMAGE_EXTENSIONS
]
if not test_paths:
    raise ValueError(f"No supported test images found in {TEST_DIR}")

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
model.load_state_dict(checkpoint["model_state"])
print(f"Using best checkpoint from epoch {checkpoint['epoch']} (validation F1 {checkpoint['val_f1']:.4f})")

test_ds = TestDataset(test_paths, val_tfms)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
model.eval()
results = []

with torch.no_grad():
    for images, names in test_loader:
        outputs = model(images.to(DEVICE))
        probabilities = torch.sigmoid(outputs).cpu().numpy().ravel()
        for name, probability in zip(names, probabilities):
            results.append((name, int(probability > 0.5)))

SUBMISSION

In [ ]:
submission = pd.DataFrame(results, columns=["ID", "Label"])
os.makedirs(os.path.dirname(SUBMISSION_PATH), exist_ok=True)
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Saved {len(submission)} predictions to {SUBMISSION_PATH}")